# Error Analysis and January Peak Feature

This notebook investigates where the enhanced Gradient Boosting model makes its largest mistakes and tests a domain-informed January returns feature.

> The dataset is synthetic and contains no confidential DHL data.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

DATA_PATH = Path("../data/dhl_style_parcel_volume.csv")
df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df = df.sort_values(["depot", "date"]).copy()

## Recreate enhanced historical features

In [ ]:
df["lag_1"] = df.groupby("depot")["parcel_volume"].shift(1)
df["lag_7"] = df.groupby("depot")["parcel_volume"].shift(7)
df["lag_14"] = df.groupby("depot")["parcel_volume"].shift(14)
df["lag_28"] = df.groupby("depot")["parcel_volume"].shift(28)

df["rolling_mean_7"] = (
    df.groupby("depot")["parcel_volume"]
      .transform(lambda x: x.shift(1).rolling(7).mean())
)

df["rolling_mean_28"] = (
    df.groupby("depot")["parcel_volume"]
      .transform(lambda x: x.shift(1).rolling(28).mean())
)

df["month"] = df["date"].dt.month
df["day_of_week_num"] = df["date"].dt.dayofweek

In [ ]:
features_v2 = [
    "lag_1", "lag_7", "lag_14", "lag_28",
    "rolling_mean_7", "rolling_mean_28",
    "day_of_week_num", "month",
    "is_bank_holiday", "is_black_friday_period",
    "is_christmas_peak",
]

model_df_v2 = df.dropna(subset=features_v2).copy()
train_df_v2 = model_df_v2[model_df_v2["date"] < "2026-01-01"].copy()
test_df_v2 = model_df_v2[model_df_v2["date"] >= "2026-01-01"].copy()

X_train_v2 = train_df_v2[features_v2]
y_train_v2 = train_df_v2["parcel_volume"]
X_test_v2 = test_df_v2[features_v2]
y_test_v2 = test_df_v2["parcel_volume"]

gb_model_v2 = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb_model_v2.fit(X_train_v2, y_train_v2)
gb_predictions_v2 = gb_model_v2.predict(X_test_v2)
gb_mae_v2 = mean_absolute_error(y_test_v2, gb_predictions_v2)

## Error analysis

In [ ]:
results_df = test_df_v2[["date", "depot", "parcel_volume"]].copy()
results_df["predicted_volume"] = gb_predictions_v2
results_df["error"] = results_df["parcel_volume"] - results_df["predicted_volume"]
results_df["absolute_error"] = results_df["error"].abs()

largest_errors = results_df.sort_values("absolute_error", ascending=False).head(10)
largest_errors

In [ ]:
results_df["month"] = results_df["date"].dt.month

monthly_errors = (
    results_df.groupby("month")["absolute_error"]
    .mean()
    .sort_values(ascending=False)
)

monthly_errors

In [ ]:
depot_errors = results_df.groupby("depot").agg(
    MAE=("absolute_error", "mean"),
    Average_Error=("error", "mean"),
    Largest_Error=("absolute_error", "max"),
).sort_values("MAE", ascending=False)

depot_errors

January has the highest monthly average absolute error, and Depot A is the hardest depot to predict. The positive average signed error for Depot A indicates a tendency to underpredict.

## Add a January returns feature

In [ ]:
df["is_january_returns_peak"] = (
    (df["date"].dt.month == 1) &
    (df["date"].dt.day <= 10)
).astype(int)

features_v3 = features_v2 + ["is_january_returns_peak"]

model_df_v3 = df.dropna(subset=features_v3).copy()
train_df_v3 = model_df_v3[model_df_v3["date"] < "2026-01-01"].copy()
test_df_v3 = model_df_v3[model_df_v3["date"] >= "2026-01-01"].copy()

X_train_v3 = train_df_v3[features_v3]
y_train_v3 = train_df_v3["parcel_volume"]
X_test_v3 = test_df_v3[features_v3]
y_test_v3 = test_df_v3["parcel_volume"]

In [ ]:
gb_model_v3 = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb_model_v3.fit(X_train_v3, y_train_v3)
gb_predictions_v3 = gb_model_v3.predict(X_test_v3)
gb_mae_v3 = mean_absolute_error(y_test_v3, gb_predictions_v3)

improvement_pct = ((gb_mae_v2 - gb_mae_v3) / gb_mae_v2) * 100

print(f"Previous Enhanced MAE: {gb_mae_v2:.2f}")
print(f"January-Aware MAE: {gb_mae_v3:.2f}")
print(f"Improvement: {improvement_pct:.2f}%")

### Result

- Previous enhanced model MAE: approximately **763.32**
- January-aware model MAE: approximately **761.13**
- Improvement: only about **0.3%**

This indicates that the January returns indicator alone explains only a small amount of the remaining forecasting error. The stronger next hypothesis is to model depot identity explicitly because error analysis shows clear depot-level performance differences.